In [1]:
import os 
import pandas as pd 
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [2]:
from MisC.utility import *

In [18]:
os.chdir("/work/DPDS/s205711/yunguan")

In [19]:
cell_by_gene_path = "./data/merscope/cell_by_gene.csv"
detected_transcripts_path = './data/merscope/detected_transcripts.csv'
cell_metadata_path = './data/merscope/cell_metadata.csv'
cell_boundaries_path = "./data/merscope/cell_boundaries.parquet"

In [20]:
adata, cell_coords, tx_metadata = import_data(cell_by_gene_counts=cell_by_gene_path,
                                              cell_metadata=cell_metadata_path,
                                              cell_boundary_polygons=cell_boundaries_path,
                                              detected_transcripts=detected_transcripts_path)

Successfully read in data. Performing basic transformation
UMAP


/home2/s205711/.conda/envs/yunguan/lib/python3.9/site-packages/scanpy/tools/_utils.py:41: UserWarning: You’re trying to run this on 400 dimensions of `.X`, if you really want this, set `use_rep='X'`.
         Falling back to preprocessing with `sc.pp.pca` and default params.
  warnings.warn(


Leiden
Done~


In [21]:
mask_distance = calculate_mask_distance(adata=adata, 
                                        cell_coords=cell_coords)

In [22]:
intf_tx = annotate_tx_mask_distance(adata=adata, 
                                    tx_metadata=tx_metadata,
                                    cell_coords=cell_coords,
                                    mask_distance=mask_distance)

In [23]:
layer = "counts_0"

In [109]:
sample1 = adata.to_df(layer).groupby(adata.obs.leiden, 
                           observed=True, 
                           as_index=False).apply(lambda x: x.sample(1000, replace=True)).reset_index(drop=False,
                                                                                    names=['leiden', 'cell_id']).drop(columns=['cell_id'])
sample2 = adata.to_df(layer).groupby(adata.obs.leiden, 
                           observed=True, 
                           as_index=False).apply(lambda x: x.sample(1000, replace=True)).reset_index(drop=False,
                                                                                    names=['leiden', 'cell_id']).drop(columns=['cell_id'])

In [110]:
counts1 = sample1.groupby(by=['leiden'], observed=True, as_index=False).sum()
counts2 = sample2.groupby(by=['leiden'], observed=True, as_index=False).sum()

In [111]:
count_df = pd.concat([counts1, counts2], axis=0).reset_index(drop=True)

In [112]:
count_df["sample_id"] = "Sample"+count_df.index.astype(str)

In [113]:
metadata = count_df[["sample_id", "leiden"]]

In [114]:
metadata.set_index("sample_id", inplace=True)

In [115]:
metadata

,leiden
sample_id,
Sample0,0
Sample1,1
Sample2,2
Sample3,3
Sample4,4
Sample5,5
Sample6,6
Sample7,0
Sample8,1


In [116]:
count_df.drop(columns=['leiden'], inplace=True)

In [117]:
count_df.set_index('sample_id', inplace=True)

In [118]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=count_df,
    metadata=metadata,
    design_factors="leiden",
    refit_cooks=True,
    inference=inference,
)


In [119]:
dds.deseq2()

Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.08 seconds.

Fitting dispersion trend curve...
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.09 seconds.

Fitting LFCs...
... done in 0.08 seconds.

Calculating cook's distance...
... done in 0.00 seconds.

Replacing 0 outlier genes.



In [120]:
stat_res = DeseqStats(dds, inference=inference, contrast=['leiden', "0",'3'],
                      quiet=True)

In [121]:
stat_res.summary()

In [135]:
((stat_res.results_df['padj'] <= 0.05) & (stat_res.results_df['log2FoldChange']>1)).sum()

71

In [89]:
s_total = adata.to_df(layer).groupby(adata.obs.leiden, observed=True).count()
s_above_0 = (adata.to_df(layer)>0).groupby(adata.obs.leiden, observed=True).sum()
percent_pos = s_above_0 / s_total

In [127]:
temp = percent_pos.loc["0", :] - percent_pos.loc['3',:]

In [131]:
temp[(temp>0.25) | (temp<-0.25)]

ADH1B       0.466489
BAAT        0.649536
BRD2        0.342728
C4BPA       0.777429
CD14        0.498919
COL1A1     -0.344000
CXCR4      -0.279085
CYP27A1     0.360939
DPT        -0.334430
FBP1        0.748546
FN1         0.419743
GSN        -0.285459
HERPUD1     0.430862
HSD17B6     0.702492
IL6ST       0.356923
ITGB1       0.303220
ITIH3       0.589306
JUN         0.259533
KLF3        0.321232
MASP2       0.670325
MGP        -0.276064
MMP2       -0.265872
NEK6        0.257776
NFKBIA      0.266966
NFKBIZ      0.257759
PDK4        0.631167
PROX1       0.664917
PSAP        0.288782
RHOB        0.475394
RORA        0.452693
SDC1        0.644740
SELENOP     0.342303
SERPIND1    0.790484
SMIM14      0.276518
SNX3        0.276740
SNX9        0.342339
SRSF5       0.276838
CEMIP2      0.251317
UGT2B4      0.546025
XBP1        0.466962
ZBTB16      0.353539
dtype: float64

In [97]:
stat_res.results_df.loc[['FOS'], :]

,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj
FOS,5.162962,-2.764172,3.219675,-0.858525,0.390603,0.983139
